In [1]:
# import necessary libraries
import pandas as pd
import numpy as np

In [2]:
# Load the raw combined energy data
data = pd.read_csv("data/energy.csv")

In [3]:
# Preview first and last rows to check data structure and spot obvious issues
data 

,Unnamed: 0,Country,Energy_type,Year,Energy_consumption,Energy_production,GDP,Population,Energy_intensity_per_capita,Energy_intensity_by_GDP,CO2_emission
0,0,World,all_energy_types,1980,292.899790,296.337228,27770.910281,4.298127e+06,68.145921,10.547000,4946.627130
1,1,World,coal,1980,78.656134,80.114194,27770.910281,4.298127e+06,68.145921,10.547000,1409.790188
2,2,World,natural_gas,1980,53.865223,54.761046,27770.910281,4.298127e+06,68.145921,10.547000,1081.593377
3,3,World,petroleum_n_other_liquids,1980,132.064019,133.111109,27770.910281,4.298127e+06,68.145921,10.547000,2455.243565
4,4,World,nuclear,1980,7.575700,7.575700,27770.910281,4.298127e+06,68.145921,10.547000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
55435,55435,Zimbabwe,coal,2019,0.045064,0.075963,37.620400,1.465420e+04,11.508701,4.482962,4.586869
55436,55436,Zimbabwe,natural_gas,2019,0.000000,0.000000,37.620400,1.465420e+04,11.508701,4.482962,0.000000
55437,55437,Zimbabwe,petroleum_n_other_liquids,2019,0.055498,0.000000,37.620400,1.465420e+04,11.508701,4.482962,4.377890
55438,55438,Zimbabwe,nuclear,2019,NaN,NaN,37.620400,1.465420e+04,11.508701,4.482962,0.000000


In [4]:
# Check for missing values and data types
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 55440 entries, 0 to 55439
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Unnamed: 0                   55440 non-null  int64  
 1   Country                      55440 non-null  str    
 2   Energy_type                  55440 non-null  str    
 3   Year                         55440 non-null  int64  
 4   Energy_consumption           44287 non-null  float64
 5   Energy_production            44289 non-null  float64
 6   GDP                          40026 non-null  float64
 7   Population                   46014 non-null  float64
 8   Energy_intensity_per_capita  50358 non-null  float64
 9   Energy_intensity_by_GDP      50358 non-null  float64
 10  CO2_emission                 51614 non-null  float64
dtypes: float64(7), int64(2), str(2)
memory usage: 4.7 MB


In [5]:
# Drop the unnecessary "Unnamed: 0" column if it exists
data = data.drop(columns=["Unnamed: 0"])

In [6]:
# NOTE: GDP and Population are NOT filled with the global median here.
# Filling with the median first would hide the gaps, so the per-country
# forward/backward fill below would never run and countries with no GDP
# would never be dropped. Missing values are handled per country further down.

In [7]:
# Remove the "World" row, as it may skew the analysis
data = data[data["Country"] != "World"]

In [8]:
# Display the data after cleaning to verify the changes
data 

,Country,Energy_type,Year,Energy_consumption,Energy_production,GDP,Population,Energy_intensity_per_capita,Energy_intensity_by_GDP,CO2_emission
6,Afghanistan,all_energy_types,1980,0.026583,0.072561,NaN,13356.5,1.990283,0.000000,NaN
7,Afghanistan,coal,1980,0.002479,0.002355,NaN,13356.5,1.990283,0.000000,NaN
8,Afghanistan,natural_gas,1980,0.002094,0.062820,NaN,13356.5,1.990283,0.000000,NaN
9,Afghanistan,petroleum_n_other_liquids,1980,0.014624,0.000000,NaN,13356.5,1.990283,0.000000,NaN
10,Afghanistan,nuclear,1980,NaN,NaN,NaN,13356.5,1.990283,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
55435,Zimbabwe,coal,2019,0.045064,0.075963,37.6204,14654.2,11.508701,4.482962,4.586869
55436,Zimbabwe,natural_gas,2019,0.000000,0.000000,37.6204,14654.2,11.508701,4.482962,0.000000
55437,Zimbabwe,petroleum_n_other_liquids,2019,0.055498,0.000000,37.6204,14654.2,11.508701,4.482962,4.377890
55438,Zimbabwe,nuclear,2019,NaN,NaN,37.6204,14654.2,11.508701,4.482962,0.000000


In [9]:
# Display summary statistics to understand the distribution of the data
data.describe()

,Year,Energy_consumption,Energy_production,GDP,Population,Energy_intensity_per_capita,Energy_intensity_by_GDP,CO2_emission
count,55200.000000,44047.000000,4.404900e+04,39786.000000,4.577400e+04,50118.000000,50118.000000,51374.000000
mean,1999.500000,0.773095,7.705254e-01,413.235917,3.126726e+04,71.909197,3.680257,39.584103
std,11.543501,4.692671,4.391539e+00,1449.007692,1.230775e+05,114.000087,4.595092,299.125633
min,1980.000000,-0.163438,-1.000000e-39,0.124958,1.147100e+01,0.000000,0.000000,-0.005130
25%,1989.750000,0.000000,0.000000e+00,9.647490,1.120020e+03,3.739487,0.880636,0.000000
50%,1999.500000,0.017761,4.436977e-04,46.842800,6.062460e+03,29.425526,2.972417,0.000000
75%,2009.250000,0.200697,1.058056e-01,254.398100,1.964405e+04,95.916542,4.927267,4.202312
max,2019.000000,151.608886,1.235914e+02,23128.340000,1.434512e+06,1139.320598,166.913605,10732.002367


In [10]:
# Handle negative values in energy consumption, production, and CO2 emissions by replacing them with NaN
for col in ["Energy_consumption", "Energy_production", "CO2_emission"]:
    data[col] = data[col].where(data[col] >= 0, other=np.nan)


In [11]:
# Investigate zero values in Energy_intensity_by_GDP
# True zeros are valid (no energy use) but zeros caused by missing consumption/GDP are not
# Result will determine whether to replace zeros with NaN
data[data["Energy_intensity_by_GDP"] == 0].groupby("Country").size()

Country
Afghanistan        48
Albania            48
Algeria            48
American Samoa    138
Angola             48
                 ... 
Wake Island       240
Western Sahara    240
Yemen              60
Zambia             48
Zimbabwe           48
Length: 147, dtype: int64

In [12]:
# Create a mask to identify rows with missing values in "Energy_consumption" or "GDP" columns
mask = (data["Energy_consumption"].isna() | data["GDP"].isna())


In [13]:
# Nullify Energy_intensity_by_GDP where source data is missing
# (cannot compute intensity without both consumption and GDP)
data.loc[mask, "Energy_intensity_by_GDP"] = np.nan

In [14]:
data.loc[mask, "Energy_intensity_per_capita"] = np.nan

In [15]:
# Remove countries with no GDP data, as they cannot be included in the analysis
countries_no_gdp = data.groupby("Country")["GDP"].apply(lambda x: x.isna().all())
countries_to_drop = countries_no_gdp[countries_no_gdp].index
data = data[~data["Country"].isin(countries_to_drop)]

In [16]:
# Check to verify that countries with no GDP data have been removed
data 

,Country,Energy_type,Year,Energy_consumption,Energy_production,GDP,Population,Energy_intensity_per_capita,Energy_intensity_by_GDP,CO2_emission
6,Afghanistan,all_energy_types,1980,0.026583,0.072561,NaN,13356.5,NaN,NaN,NaN
7,Afghanistan,coal,1980,0.002479,0.002355,NaN,13356.5,NaN,NaN,NaN
8,Afghanistan,natural_gas,1980,0.002094,0.062820,NaN,13356.5,NaN,NaN,NaN
9,Afghanistan,petroleum_n_other_liquids,1980,0.014624,0.000000,NaN,13356.5,NaN,NaN,NaN
10,Afghanistan,nuclear,1980,NaN,NaN,NaN,13356.5,NaN,NaN,0.000000
...,...,...,...,...,...,...,...,...,...,...
55435,Zimbabwe,coal,2019,0.045064,0.075963,37.6204,14654.2,11.508701,4.482962,4.586869
55436,Zimbabwe,natural_gas,2019,0.000000,0.000000,37.6204,14654.2,11.508701,4.482962,0.000000
55437,Zimbabwe,petroleum_n_other_liquids,2019,0.055498,0.000000,37.6204,14654.2,11.508701,4.482962,4.377890
55438,Zimbabwe,nuclear,2019,NaN,NaN,37.6204,14654.2,NaN,NaN,0.000000


In [17]:
# Fill GDP and Population within each country (forward then backward fill)
# This is better than global median since these values change gradually per country
data["GDP"] = data.groupby("Country")["GDP"].transform(lambda x: x.ffill().bfill())
data["Population"] = data.groupby("Country")["Population"].transform(lambda x: x.ffill().bfill())


In [18]:
# For any countries with no GDP/Population data at all, fall back to global median
data["GDP"] = data["GDP"].fillna(data["GDP"].median())
data["Population"] = data["Population"].fillna(data["Population"].median())

In [19]:
# Forward/back fill consumption and production within each country-energy_type group
data["Energy_consumption"] = data.groupby(["Country", "Energy_type"])["Energy_consumption"].transform(lambda x: x.ffill().bfill())
data["Energy_production"] = data.groupby(["Country", "Energy_type"])["Energy_production"].transform(lambda x: x.ffill().bfill())

In [20]:
# Recompute intensity columns from cleaned source data
# Population divided by 1000 to match units (energy per thousand people)
# Recomputing ensures consistency with filled GDP and consumption values
data["Energy_intensity_per_capita"] = data["Energy_consumption"] / (data["Population"] / 1000)
data["Energy_intensity_by_GDP"] = data["Energy_consumption"] / data["GDP"]

In [21]:
#  Drop rows where consumption/production couldn't be filled
# These are countries with no data across all years for a given energy type
data = data.dropna(subset=["Energy_consumption", "Energy_production"])

In [22]:
# Rename columns for better readability in visualizations
data = data.rename(columns={
    "Energy_consumption": "Energy_Consumption",
    "Energy_production": "Energy_Production",
    "Energy_intensity_per_capita": "Energy_Intensity_Per_Capita",
    "Energy_intensity_by_GDP": "Energy_Intensity_By_GDP",
    "CO2_emission": "CO2_Emission",
    "Energy_type": "Energy_Type"
})

In [23]:
# Null counts as a table
display(data.isnull().sum().reset_index().rename(columns={"index": "Column", 0: "Null Count"}))

# Summary statistics as a table
display(data.describe())

,Column,Null Count
0,Country,0
1,Energy_Type,0
2,Year,0
3,Energy_Consumption,0
4,Energy_Production,0
5,GDP,0
6,Population,0
7,Energy_Intensity_Per_Capita,0
8,Energy_Intensity_By_GDP,0
9,CO2_Emission,2687


,Year,Energy_Consumption,Energy_Production,GDP,Population,Energy_Intensity_Per_Capita,Energy_Intensity_By_GDP,CO2_Emission
count,41920.000000,41920.000000,41920.000000,41920.000000,4.192000e+04,41920.000000,41920.000000,39233.000000
mean,1999.500000,0.809064,0.804513,394.332776,3.293118e+04,0.036265,0.002224,48.467753
std,11.543534,4.732306,4.401833,1419.561396,1.282590e+05,0.099788,0.005974,332.509261
min,1980.000000,0.000000,0.000000,0.124958,1.147100e+01,0.000000,0.000000,0.000000
25%,1989.750000,0.000510,0.000000,8.689150,1.314533e+03,0.000108,0.000022,0.000000
50%,1999.500000,0.029433,0.002147,38.824900,6.160695e+03,0.005156,0.000812,0.300000
75%,2009.250000,0.242554,0.135464,227.840300,1.999650e+04,0.033186,0.002568,9.662105
max,2019.000000,151.608886,123.591360,23128.340000,1.434512e+06,2.102356,0.166914,10732.002367


In [24]:
# Display the final cleaned
data

,Country,Energy_Type,Year,Energy_Consumption,Energy_Production,GDP,Population,Energy_Intensity_Per_Capita,Energy_Intensity_By_GDP,CO2_Emission
6,Afghanistan,all_energy_types,1980,0.026583,0.072561,37.7013,13356.5,0.001990,0.000705,NaN
7,Afghanistan,coal,1980,0.002479,0.002355,37.7013,13356.5,0.000186,0.000066,NaN
8,Afghanistan,natural_gas,1980,0.002094,0.062820,37.7013,13356.5,0.000157,0.000056,NaN
9,Afghanistan,petroleum_n_other_liquids,1980,0.014624,0.000000,37.7013,13356.5,0.001095,0.000388,NaN
11,Afghanistan,renewables_n_other,1980,0.007386,0.007386,37.7013,13356.5,0.000553,0.000196,0.000000
...,...,...,...,...,...,...,...,...,...,...
55434,Zimbabwe,all_energy_types,2019,0.168651,0.143462,37.6204,14654.2,0.011509,0.004483,8.964759
55435,Zimbabwe,coal,2019,0.045064,0.075963,37.6204,14654.2,0.003075,0.001198,4.586869
55436,Zimbabwe,natural_gas,2019,0.000000,0.000000,37.6204,14654.2,0.000000,0.000000,0.000000
55437,Zimbabwe,petroleum_n_other_liquids,2019,0.055498,0.000000,37.6204,14654.2,0.003787,0.001475,4.377890


In [25]:
# Convert "Year" column to datetime format for better handling in visualizations
data["Year"] = pd.to_datetime(data["Year"], format="%Y")

In [26]:
# Save the cleaned data to a new CSV file for use in visualizations
data.to_csv("data/energy_cleaned.csv", index=False)